[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C64_ML_Knowledge_QA_Course/04_eval_stats/04_eval_stats_qa.ipynb)

# 04 · 评估、概率与统计问答（指标全家 / ROC vs PR-AUC / 校准 / 置信区间 / A/B 测试 / 统计陷阱）

目标：把「这个指标什么时候会骗你」从口头描述，变成**能跑出数字、能亲眼看到分歧**的小实验。

本 notebook 你会亲手实现：
1. **环境自检**
2. **指标全家计算器** —— 从混淆矩阵算 Accuracy/Precision/Recall/F1/Fβ，并验证它们之间的代数关系
3. **ROC-AUC vs PR-AUC 分歧演示** —— 同一个分类器，在平衡与不平衡数据上对比两种 AUC
4. **代价敏感阈值选择** —— 给定代价矩阵，扫描阈值求期望代价最小的工作点
5. **校准：可靠性图 / ECE / 温度缩放** —— 亲手把一个「过度自信」的模型校准回去
6. **三种置信区间对比** —— 正态 / bootstrap / Wilson，小样本下看谁更稳
7. **A/B 测试的样本量与功效** —— 反推「要检出这个效应需要多少用户」
8. **辛普森悖论的可复现构造** —— 分组内都占优，合并后反转

> 心智模型：**统计方法的正确性不是「公式对不对」，是「前提假设成不成立」。**

## 0 · 环境自检

本课全程只用标准库 + numpy。没有 GPU 依赖、不联网、不下载数据。

In [ ]:
import sys, math, random
import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)

assert sys.version_info >= (3, 8), '需要 Python 3.8+'
assert hasattr(np, 'trapezoid') or hasattr(np, 'trapz')

rng_global = np.random.default_rng(0)
print('\n✅ 环境自检通过：本模块不需要 GPU、不需要联网，全程固定 seed 保证可复现。')

## 1 · 指标全家计算器

从原始的 TP/FP/FN/TN 出发，算出 Accuracy/Precision/Recall/F1/Fβ，
并验证「F1 是 Fβ 在 β=1 时的特例」「β→0 退化为 Precision，β→∞ 退化为 Recall」。

In [ ]:
def confusion_counts(y_true, y_pred):
    """y_true/y_pred: 0/1 数组，返回 (TP, FP, FN, TN)。"""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    return tp, fp, fn, tn

def precision_recall(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return p, r

def f_beta(p, r, beta=1.0):
    num = (1 + beta**2) * p * r
    den = beta**2 * p + r
    return num / den if den > 0 else 0.0

# 构造一组带明显不平衡的预测
y_true = [1]*10 + [0]*90
y_pred = [1]*9 + [0]*1 + [1]*10 + [0]*80     # 9 个真阳性、10 个假阳性、1 个假阴性

tp, fp, fn, tn = confusion_counts(y_true, y_pred)
p, r = precision_recall(tp, fp, fn)
acc = (tp + tn) / len(y_true)

assert (tp, fp, fn, tn) == (9, 10, 1, 80)
assert abs(p - 9/19) < 1e-9 and abs(r - 0.9) < 1e-9
assert abs(acc - 0.89) < 1e-9

f1 = f_beta(p, r, 1.0)
f05 = f_beta(p, r, 0.5)
f2 = f_beta(p, r, 2.0)
assert abs(f1 - 2*p*r/(p+r)) < 1e-9
assert f05 < f1 < f2                          # beta 越大越贴近 recall（这里 recall > precision）
assert abs(f_beta(p, r, 1e-6) - p) < 1e-4      # beta -> 0 退化为 precision
assert abs(f_beta(p, r, 1e6) - r) < 1e-4       # beta -> inf 退化为 recall

print(f'TP={tp} FP={fp} FN={fn} TN={tn}   accuracy={acc:.3f}')
print(f'precision={p:.3f}  recall={r:.3f}')
print(f'F0.5={f05:.3f}  F1={f1:.3f}  F2={f2:.3f}   (recall > precision -> beta 越大分数越高)')
print('\n✅ 指标全家验证通过：Fβ 在 β→0/1/∞ 三个极限下都符合预期。')

## 2 · ROC-AUC vs PR-AUC 分歧演示

同一个分类器（用分数排序模拟），分别放到「平衡」和「不平衡」两份数据上，
看两种 AUC 谁先「露馅」。

In [ ]:
def roc_auc(y_true, scores):
    """ROC-AUC = P(score(正例) > score(负例))，用 Mann-Whitney U 的等价形式算，避免依赖 sklearn。"""
    y_true = np.asarray(y_true); scores = np.asarray(scores)
    pos = scores[y_true == 1]; neg = scores[y_true == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float('nan')
    # 对每个正例，统计有多少负例分数更低（打平记 0.5）
    greater = (pos[:, None] > neg[None, :]).sum()
    equal = (pos[:, None] == neg[None, :]).sum()
    return (greater + 0.5 * equal) / (len(pos) * len(neg))

def pr_curve(y_true, scores):
    """按分数降序扫描阈值，返回 (recall_list, precision_list)，用于算 PR-AUC。"""
    y_true = np.asarray(y_true)
    order = np.argsort(-scores)
    y_sorted = y_true[order]
    tp_cum = np.cumsum(y_sorted == 1)
    fp_cum = np.cumsum(y_sorted == 0)
    n_pos = int((y_true == 1).sum())
    recall = tp_cum / max(n_pos, 1)
    precision = tp_cum / np.maximum(tp_cum + fp_cum, 1)
    return recall, precision

def pr_auc(y_true, scores):
    """用梯形法则对 (recall, precision) 积分，recall 需先排序。"""
    recall, precision = pr_curve(y_true, scores)
    order = np.argsort(recall)
    r_sorted, p_sorted = recall[order], precision[order]
    return float(np.trapezoid(p_sorted, r_sorted)) if hasattr(np, 'trapezoid') \
        else float(np.trapz(p_sorted, r_sorted))

rng = np.random.default_rng(42)

def make_scored_data(n_pos, n_neg, sep=1.2):
    """正例分数 ~ N(sep, 1)，负例分数 ~ N(0, 1)，sep 控制可分性（两份数据用同一个 sep，即"同一个分类器"）。"""
    pos_scores = rng.normal(sep, 1.0, n_pos)
    neg_scores = rng.normal(0.0, 1.0, n_neg)
    y = np.array([1]*n_pos + [0]*n_neg)
    s = np.concatenate([pos_scores, neg_scores])
    return y, s

# 平衡数据：500 正 500 负
y_bal, s_bal = make_scored_data(500, 500)
auc_bal = roc_auc(y_bal, s_bal)
prauc_bal = pr_auc(y_bal, s_bal)

# 不平衡数据：同样的分类器能力（sep 相同），但正例只有 1%
y_imb, s_imb = make_scored_data(100, 9900)
auc_imb = roc_auc(y_imb, s_imb)
prauc_imb = pr_auc(y_imb, s_imb)

print(f'{"数据集":<10}{"ROC-AUC":>10}{"PR-AUC":>10}')
print(f'{"平衡 1:1":<10}{auc_bal:>10.3f}{prauc_bal:>10.3f}')
print(f'{"不平衡 1:99":<10}{auc_imb:>10.3f}{prauc_imb:>10.3f}')

# 核心断言：分类器能力不变，ROC-AUC 几乎不受类别比例影响；PR-AUC 大幅下降
assert abs(auc_bal - auc_imb) < 0.03, 'ROC-AUC 应该对类别比例基本不敏感'
assert prauc_imb < prauc_bal - 0.15, 'PR-AUC 应该随正类稀释明显下降'
print('\n✅ 分歧验证通过：同一个分类器，ROC-AUC 几乎不变，PR-AUC 明显下降 —— 这就是"ROC-AUC 会骗人"的数值证据。')

## 3 · 代价敏感阈值选择

给定 $C_{FP}, C_{FN}$，扫描阈值求期望代价最小的工作点，而不是无脑用 0.5。

In [ ]:
def best_threshold(y_true, scores, c_fp, c_fn, grid=None):
    """扫描阈值网格，返回 (最优阈值, 最小期望代价, 完整记录)。"""
    y_true = np.asarray(y_true); scores = np.asarray(scores)
    if grid is None:
        grid = np.linspace(scores.min(), scores.max(), 200)
    best_t, best_cost, records = None, float('inf'), []
    for t in grid:
        pred = (scores >= t).astype(int)
        tp, fp, fn, tn = confusion_counts(y_true, pred)
        cost = c_fp * fp + c_fn * fn
        records.append((t, cost, fp, fn))
        if cost < best_cost:
            best_cost, best_t = cost, t
    return best_t, best_cost, records

y_imb2, s_imb2 = y_imb, s_imb   # 复用上面的不平衡数据

# 场景一：漏检代价远高于误检（例如 TSR 停车让行标志）
t_a, cost_a, _ = best_threshold(y_imb2, s_imb2, c_fp=1.0, c_fn=20.0)
# 场景二：两种错误代价相同
t_b, cost_b, _ = best_threshold(y_imb2, s_imb2, c_fp=1.0, c_fn=1.0)

assert t_a < t_b, '漏检代价越高，最优阈值应该越低（更愿意多报警）'
print(f'漏检代价高（C_FN=20） -> 最优阈值 {t_a:.3f}，期望代价 {cost_a:.1f}')
print(f'两种代价相同（C_FN=1）  -> 最优阈值 {t_b:.3f}，期望代价 {cost_b:.1f}')
print(f'\n阈值差 = {t_b - t_a:.3f} > 0，验证「漏检代价越高，阈值应该越低」这条方向性结论。')
print('✅ 代价敏感阈值选择通过：阈值不是"读"出来的，是从代价矩阵"算"出来的。')

## 4 · 校准：可靠性图 / ECE / 温度缩放

先构造一个「刻意过度自信」的模型输出，量化它的 ECE，再用温度缩放把它校准回去。

In [ ]:
def make_overconfident_probs(y_true, base_acc=0.75, rng=rng):
    """构造一个"实际准确率只有 base_acc，但汇报的置信度普遍很高"的模型输出。
    关键是让 preds 的正确性由 base_acc 独立决定，汇报的置信度与真实正确性脱钩——
    这正是"过度自信"的本质：置信度没有跟着真实难度走。"""
    y_true = np.asarray(y_true)
    n = len(y_true)
    is_correct = rng.random(n) < base_acc
    preds = np.where(is_correct, y_true, 1 - y_true)          # 按 base_acc 决定预测对不对
    conf_reported = np.clip(1 - 0.5 * rng.beta(1, 6, n), 0.55, 0.999)   # 汇报的置信度普遍偏高（0.55~0.999）
    probs = np.where(preds == 1, conf_reported, 1 - conf_reported)
    return probs

def reliability_bins(y_true, probs, n_bins=10):
    """按置信度分桶，返回每桶的 (平均置信度, 实际准确率, 样本数)。"""
    y_true = np.asarray(y_true); probs = np.asarray(probs)
    preds = (probs >= 0.5).astype(int)
    correct = (preds == y_true).astype(float)
    conf = np.where(preds == 1, probs, 1 - probs)      # "预测那个类"的置信度
    edges = np.linspace(0, 1, n_bins + 1)
    out = []
    for i in range(n_bins):
        lo, hi = edges[i], edges[i+1]
        mask = (conf > lo) & (conf <= hi) if i > 0 else (conf >= lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        out.append((conf[mask].mean(), correct[mask].mean(), int(mask.sum())))
    return out

def ece(bins, n_total):
    return sum(cnt / n_total * abs(acc - cf) for cf, acc, cnt in bins)

y_true3 = rng.integers(0, 2, 2000)
probs_over = make_overconfident_probs(y_true3)

bins_over = reliability_bins(y_true3, probs_over)
ece_over = ece(bins_over, len(y_true3))
print('过度自信模型的可靠性桶（置信度, 准确率, 样本数）：')
for cf, acc, cnt in bins_over:
    print(f'  conf={cf:.2f}  acc={acc:.2f}  n={cnt}')
print(f'ECE(过度自信) = {ece_over:.3f}')
assert ece_over > 0.08, '构造的模型应该明显过度自信'

In [ ]:
def apply_temperature(probs, T):
    """把概率转回 logit，除以 T，再转回概率 —— 温度缩放不改变预测排序，只压平置信度。"""
    probs = np.clip(probs, 1e-6, 1 - 1e-6)
    logit = np.log(probs / (1 - probs))
    scaled_logit = logit / T
    return 1 / (1 + np.exp(-scaled_logit))

def fit_temperature(y_true, probs, T_grid=None):
    """在验证集上网格搜索使 ECE 最小的 T（真实工程里常用 NLL 而不是 ECE，这里为了直接讲清意图用 ECE）。"""
    if T_grid is None:
        T_grid = np.linspace(0.5, 8.0, 60)
    best_T, best_ece = 1.0, float('inf')
    for T in T_grid:
        p_scaled = apply_temperature(probs, T)
        b = reliability_bins(y_true, p_scaled)
        e = ece(b, len(y_true))
        if e < best_ece:
            best_ece, best_T = e, T
    return best_T, best_ece

T_star, ece_after = fit_temperature(y_true3, probs_over)
preds_before = (probs_over >= 0.5).astype(int)
p_scaled = apply_temperature(probs_over, T_star)
preds_after = (p_scaled >= 0.5).astype(int)

assert T_star > 1.5, '过度自信的模型应该需要 T > 1 来压平置信度'
assert ece_after < ece_over * 0.5, '温度缩放后 ECE 应该显著下降'
assert np.array_equal(preds_before, preds_after), '温度缩放不改变预测类别（排序不变），只改变置信度数值'

print(f'最优温度 T* = {T_star:.2f}')
print(f'ECE：{ece_over:.3f}  ->  {ece_after:.3f}（下降 {(1 - ece_after/ece_over)*100:.0f}%）')
print('预测类别在缩放前后完全一致：', np.array_equal(preds_before, preds_after))
print('\n✅ 校准验证通过：温度缩放只调整"自信程度"，不改变"猜哪个类"。')

## 5 · 三种置信区间对比：正态 / bootstrap / Wilson

在同一批数据上算三种 95% 置信区间，重点看**小样本 + 极端比例**下三者的差异。

In [ ]:
def normal_ci(k, n, z=1.96):
    p = k / n
    se = math.sqrt(p * (1 - p) / n) if n > 0 else 0.0
    return max(p - z*se, -math.inf), min(p + z*se, math.inf)   # 故意不裁剪到 [0,1]，暴露它会越界

def wilson_ci(k, n, z=1.96):
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    margin = (z / denom) * math.sqrt(p*(1-p)/n + z**2/(4*n**2))
    return center - margin, center + margin

def bootstrap_ci(successes_array, n_boot=5000, z_pct=(2.5, 97.5), rng=rng):
    """successes_array: 0/1 数组。对均值做非参数 bootstrap。"""
    n = len(successes_array)
    boot_means = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_means[i] = successes_array[idx].mean()
    lo, hi = np.percentile(boot_means, z_pct)
    return float(lo), float(hi)

# 场景：小样本、高比例 —— 三者分歧最大的地方
n_small, k_small = 20, 19          # 20 次里对了 19 次
arr_small = np.array([1]*k_small + [0]*(n_small-k_small))

lo_n, hi_n = normal_ci(k_small, n_small)
lo_w, hi_w = wilson_ci(k_small, n_small)
lo_b, hi_b = bootstrap_ci(arr_small)

print(f'小样本 n={n_small}, k={k_small} (p_hat={k_small/n_small:.2f})：')
print(f'  正态   CI: [{lo_n:.3f}, {hi_n:.3f}]  <- 注意上界是否超过 1')
print(f'  Wilson CI: [{lo_w:.3f}, {hi_w:.3f}]  <- 应天然落在 [0,1] 内')
print(f'  Bootstrap CI: [{lo_b:.3f}, {hi_b:.3f}]')

assert hi_n > 1.0, '正态近似在极端比例小样本下应该越界（这正是它的失效证据）'
assert 0.0 <= lo_w and hi_w <= 1.0, 'Wilson 区间天然落在 [0,1] 内'
assert 0.0 <= lo_b <= hi_b <= 1.0

# 场景：大样本、适中比例 —— 三者应该基本一致
n_big, k_big = 5000, 2500
arr_big = np.array([1]*k_big + [0]*(n_big-k_big))
lo_n2, hi_n2 = normal_ci(k_big, n_big)
lo_w2, hi_w2 = wilson_ci(k_big, n_big)
assert abs(lo_n2 - lo_w2) < 0.01 and abs(hi_n2 - hi_w2) < 0.01, '大样本适中比例下正态与 Wilson 应几乎重合'

print(f'\n大样本 n={n_big}, p_hat=0.50：正态 [{lo_n2:.3f},{hi_n2:.3f}] vs Wilson [{lo_w2:.3f},{hi_w2:.3f}]  <- 几乎重合')
print('\n✅ 三种 CI 验证通过：小样本极端比例下正态近似会越界，Wilson 稳；大样本下三者趋同。')

## 6 · A/B 测试：样本量与功效计算

反推「要检出这个效应，需要多少用户」，并验证"效应减半、样本量翻 4 倍"这条平方反比关系。

In [ ]:
from math import erf, sqrt

def norm_cdf(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

def z_from_alpha(alpha_two_sided=0.05):
    """双侧检验的临界值，用二分查找反解正态分布分位数（不依赖 scipy）。"""
    target = 1 - alpha_two_sided / 2
    lo, hi = 0.0, 6.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if norm_cdf(mid) < target:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

def sample_size_two_proportion(p1, p2, alpha=0.05, power=0.8):
    """两比例 z 检验所需的每组样本量（正态近似公式）。"""
    z_alpha = z_from_alpha(alpha)
    z_beta = z_from_alpha(2 * (1 - power))     # 单侧功效对应的 z_beta：power=1-beta -> beta=1-power
    numerator = (z_alpha + z_beta)**2 * (p1*(1-p1) + p2*(1-p2))
    denominator = (p1 - p2)**2
    return math.ceil(numerator / denominator)

z95 = z_from_alpha(0.05)
assert abs(z95 - 1.96) < 0.01, z95      # 验证反解出的临界值就是熟悉的 1.96

n1 = sample_size_two_proportion(0.10, 0.12)          # 基线 10% -> 12%，绝对提升 2pp
n2 = sample_size_two_proportion(0.10, 0.11)          # 效应减半：绝对提升只有 1pp

print(f'z(alpha=0.05, 双侧) = {z95:.3f}')
print(f'检出 10%->12% 需要每组约 {n1:,} 用户')
print(f'检出 10%->11%（效应减半）需要每组约 {n2:,} 用户')
print(f'样本量之比 = {n2/n1:.2f}   <- 效应减半，样本量应接近翻 4 倍（平方反比）')

assert 3.5 < n2 / n1 < 4.6, (n1, n2)
print('\n✅ 功效计算验证通过：效应减半，所需样本量接近翻 4 倍。')

## 7 · 辛普森悖论的可复现构造

固定构造一组"分组内都是方案乙更优、合并后方案甲反而更优"的数据，逐行验证反转确实发生。

In [ ]:
# 组 A（小分母，方案乙表现好）与组 B（大分母，方案乙表现差），但方案乙把大量样本堆在了"容易"的组 A
group_A = {'甲': (8, 10), '乙': (45, 50)}     # (成功数, 总数)
group_B = {'甲': (30, 90), '乙': (2, 10)}

def rate(k, n):
    return k / n

rate_A_jia, rate_A_yi = rate(*group_A['甲']), rate(*group_A['乙'])
rate_B_jia, rate_B_yi = rate(*group_B['甲']), rate(*group_B['乙'])

k_jia = group_A['甲'][0] + group_B['甲'][0]; n_jia = group_A['甲'][1] + group_B['甲'][1]
k_yi = group_A['乙'][0] + group_B['乙'][0]; n_yi = group_A['乙'][1] + group_B['乙'][1]
rate_overall_jia, rate_overall_yi = rate(k_jia, n_jia), rate(k_yi, n_yi)

print(f'组 A：甲 {rate_A_jia:.2f}   乙 {rate_A_yi:.2f}   (乙更优: {rate_A_yi > rate_A_jia})')
print(f'组 B：甲 {rate_B_jia:.2f}   乙 {rate_B_yi:.2f}   (甲更优: {rate_B_jia > rate_B_yi})')
print(f'合并：甲 {rate_overall_jia:.2f}   乙 {rate_overall_yi:.2f}   (乙更优: {rate_overall_yi > rate_overall_jia})')

# 核心断言：两个分组内的"谁更优"结论不一致，且合并结果与其中至少一组的结论相反
assert rate_A_yi > rate_A_jia          # 组 A 内乙更优
assert rate_B_jia > rate_B_yi          # 组 B 内甲更优
assert rate_overall_yi > rate_overall_jia   # 合并后乙更优 —— 与组 B 内的结论相反！

# 权重解释：乙在组 A（分母小、比例高）投入的样本占比 远高于 在组 B
weight_yi_in_A = group_A['乙'][1] / n_yi
weight_jia_in_A = group_A['甲'][1] / n_jia
assert weight_yi_in_A > weight_jia_in_A
print(f'\n乙投入组 A 的样本占比 {weight_yi_in_A:.2f}，甲投入组 A 的样本占比 {weight_jia_in_A:.2f}')
print('乙把更多样本堆在了"更容易"的组 A —— 这正是合并比例被权重主导、而非被"真实优劣"主导的机制。')
print('\n✅ 辛普森悖论构造验证通过：分组内结论相反，合并结论由样本权重主导。')

## ✏️ 练习 1：Wilson 区间的宽度反推

实现 `wilson_width(k, n, z=1.96)`，直接返回 Wilson 区间的**总宽度**（上界 - 下界），
并验证「$n$ 越大，区间越窄；$\hat p$ 越极端（越接近 0 或 1），同样 $n$ 下区间越窄」。

In [ ]:
def wilson_width(k, n, z=1.96):
    # TODO: 复用 wilson_ci，返回宽度
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
w1 = wilson_width(50, 100)     # p_hat = 0.5，最不确定的比例
w2 = wilson_width(50, 400)     # 样本量翻 4 倍
w3 = wilson_width(95, 100)     # 同样 n=100，但 p_hat=0.95（更极端）

assert w2 < w1, 'n 越大，区间应该越窄'
assert w3 < w1, '同样 n 下，p_hat 越极端，区间应该越窄（方差 p(1-p) 更小）'
assert w1 > 0 and w2 > 0 and w3 > 0
print(f'n=100,  p_hat=0.50 : 宽度 {w1:.4f}')
print(f'n=400,  p_hat=0.50 : 宽度 {w2:.4f}  <- 样本翻 4 倍变窄')
print(f'n=100,  p_hat=0.95 : 宽度 {w3:.4f}  <- 比例更极端也变窄')
print('\n✅ 练习 1 通过：置信区间宽度由 n 和 p_hat(1-p_hat) 共同决定，不是只看 n。')

## ✏️ 练习 2：多重比较校正（Bonferroni）

实现 `bonferroni_correct(p_values, alpha=0.05)`，返回 `(corrected_alpha, [是否显著, ...])`。
规则：`corrected_alpha = alpha / len(p_values)`；某个 `p_value <= corrected_alpha` 才算显著。

In [ ]:
def bonferroni_correct(p_values, alpha=0.05):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
pvals = [0.001, 0.03, 0.04, 0.20, 0.049]
corrected_alpha, sig = bonferroni_correct(pvals, alpha=0.05)

assert abs(corrected_alpha - 0.01) < 1e-9
assert sig == [True, False, False, False, False], sig     # 只有 0.001 <= 0.01

# 不做校正的话，0.03/0.04/0.049 都会被判"显著"，5 个指标里假阳性概率被推高到接近 1-0.95^5≈0.226
naive_sig = [p <= 0.05 for p in pvals]
assert sum(naive_sig) == 4 and sum(sig) == 1
print(f'校正后的 alpha = {corrected_alpha}')
print(f'不校正判定显著个数: {sum(naive_sig)}   校正后判定显著个数: {sum(sig)}')
print('\n✅ 练习 2 通过：5 个指标同时检验时，不做校正会把假阳性率推到远高于 0.05。')

## ✏️ 练习 3：期望代价最小阈值的整数网格版

实现 `min_cost_threshold_int(y_true, scores_int, c_fp, c_fn)`：`scores_int` 是**已排序去重**的整数分数候选阈值列表，
对每个候选阈值 `t`（`pred = 1 当 score >= t`）算期望代价，返回 `(最优阈值, 最小代价)`；
若有并列最小代价，取**较大**的阈值（更保守，误检更少）。

In [ ]:
def min_cost_threshold_int(y_true, scores, thresholds, c_fp, c_fn):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
y_ex = [1, 1, 1, 0, 0, 0, 0, 0]
s_ex = [9, 7, 3, 8, 6, 4, 2, 1]
cand = sorted(set(s_ex))

t_opt, cost_opt = min_cost_threshold_int(y_ex, s_ex, cand, c_fp=1.0, c_fn=1.0)
# 手工核对（y=1 的分数是 {9,7,3}，y=0 的分数是 {8,6,4,2,1}）：
#   t=7 -> TP=2(9,7),FP=1(8),FN=1(3)  cost=2
#   t=8 -> TP=1(9),  FP=1(8),FN=2(7,3) cost=3
#   t=9 -> TP=1(9),  FP=0,   FN=2(7,3) cost=2  <- 与 t=7 并列最小，取更大的阈值
assert t_opt == 9, t_opt
assert abs(cost_opt - 2.0) < 1e-9, cost_opt

t_opt2, cost_opt2 = min_cost_threshold_int(y_ex, s_ex, cand, c_fp=1.0, c_fn=10.0)
assert t_opt2 < t_opt, '漏检代价升高后，最优阈值应该更低（更愿意多报警而不是漏检）'
print(f'代价相同    -> 最优阈值 {t_opt}，代价 {cost_opt}')
print(f'漏检代价升高 -> 最优阈值 {t_opt2}，代价 {cost_opt2}')
print('\n✅ 练习 3 通过：用整数网格重新验证"代价敏感阈值"，并列时选更保守的阈值。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def wilson_width(k, n, z=1.96):
    lo, hi = wilson_ci(k, n, z)
    return hi - lo

In [ ]:
# 练习 2 参考答案
def bonferroni_correct(p_values, alpha=0.05):
    corrected_alpha = alpha / len(p_values)
    sig = [p <= corrected_alpha for p in p_values]
    return corrected_alpha, sig

In [ ]:
# 练习 3 参考答案
def min_cost_threshold_int(y_true, scores, thresholds, c_fp, c_fn):
    y_true = np.asarray(y_true); scores = np.asarray(scores)
    best_t, best_cost = None, float('inf')
    for t in thresholds:
        pred = (scores >= t).astype(int)
        tp, fp, fn, tn = confusion_counts(y_true, pred)
        cost = c_fp * fp + c_fn * fn
        if cost < best_cost or (cost == best_cost and (best_t is None or t > best_t)):
            best_cost, best_t = cost, t
    return best_t, best_cost

---
## 🧪 真实工程胶囊：评估与统计的面试速查卡

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 指标选择决策树（30 秒内说完）
# ══════════════════════════════════════════════════════════════════════
# 类别是否严重不平衡？
#   是 -> 看 PR-AUC，不要只看 ROC-AUC；报告时配一条"正类比例是多少"
#   否 -> ROC-AUC 可信；两者结论应该基本一致
# 是多类检测/分类？
#   -> 报 mAP，但必须配一张逐类 AP 表（mAP 会掩盖表现极差的类，见 C61-02 TIDE 分解）
# 犯两种错误的代价是否相等？
#   否 -> 别用默认阈值 0.5，从代价矩阵算最优工作点
#   是 -> Youden's J 或 F1 最大点是合理默认

# ══════════════════════════════════════════════════════════════════════
# B. 校准三问（面试官问"模型输出的概率能信吗"时用）
# ══════════════════════════════════════════════════════════════════════
# 1) 画过可靠性图 / 算过 ECE 吗？—— 没算过，先承认"没有校准过，置信度只能当排序用，不能当概率用"
# 2) 用什么校准？—— 温度缩放优先（不改变排序，一个参数，验证集上几秒拟合完）
# 3) 校准会不会过期？—— 会，分布漂移后需要重新校准，且理想上应按场景分桶校准

# ══════════════════════════════════════════════════════════════════════
# C. "+0.3 算不算提升"的标准答法（呼应 C61-01）
# ══════════════════════════════════════════════════════════════════════
# 「不看单次数字，看多种子的方差范围；同一测试集上用配对检验（因为难度天然相关）；
#   如果只跑了一个种子，我会说"暂时无法判断是信号还是噪声，需要多种子复核"。」

# ══════════════════════════════════════════════════════════════════════
# D. A/B 测试的四步检查顺序（别跳步）
# ══════════════════════════════════════════════════════════════════════
# 1) 先查 SRM（样本比例是否符合预期分流）
# 2) 再查功效是否够（样本量是否达到 power=0.8 的要求）
# 3) 再查是否多个指标/多个时间点在偷看（需要 Bonferroni/FDR 校正，或预先固定唯一主指标）
# 4) 最后才谈论 p 值本身

# ══════════════════════════════════════════════════════════════════════
# E. 统计陷阱一句话识别表
# ══════════════════════════════════════════════════════════════════════
# 整体涨了但分组看很怪         -> 辛普森悖论，按关键维度切片复核
# 数据只来自"成功上报"的样本   -> 幸存者偏差，检查数据收集流程本身
# 极端表现之后自然回落         -> 回归到均值，需要对照组
# 试了很多种切法只报好看的那个 -> p-hacking，预注册假设 + 多重比较校正

# ══════════════════════════════════════════════════════════════════════
# F. 与本课程其他部分的分工（别重复准备）
# ══════════════════════════════════════════════════════════════════════
# · 贝叶斯公式/交叉验证的完整数学推导        -> C07（本模块只讲"怎么讲清楚"）
# · 类别不平衡的处理谱系（重采样/重加权）    -> C58-01
# · 安全导向的分桶评测体系                  -> C55-05
# · mAP/TIDE 误差分解的检测专项细节         -> C18、C53-C61
# · 种子方差与显著性的工程实践              -> C61-01
# · 150+ 题快问快答题库（含本模块全部主题） -> C64 模块 05
'''
print(RECIPE)
for token in ['PR-AUC', '温度缩放', 'SRM', '辛普森悖论', 'C61-01', 'Youden']:
    assert token in RECIPE, token
print('✅ 速查卡覆盖：指标决策树 / 校准三问 / 提升判定 / A/B 四步顺序 / 陷阱识别 / 课程分工')

### 小结

- **统计方法的正确性不是"公式对不对"，是"前提假设成不成立"。** 这是本模块所有小节的统一读法，
  也是「一句话定义 → 为什么需要 → **什么时候失效**」三段式在这一段被放大的原因。
- **ROC-AUC 在不平衡数据下会骗人**：它是一个排序统计量，分母里有海量真阴性稀释假阳性的影响；
  正类稀少时应该看 **PR-AUC**，报 mAP 时永远要配一张逐类 AP 表。
- **阈值不是"读"出来的，是从代价矩阵"算"出来的**——期望代价 $C_{FP}\cdot FP + C_{FN}\cdot FN$ 最小的点才是合理工作点。
- **高置信不等于高准确**：现代深度网络普遍过度自信，温度缩放是低成本的默认修复手段，
  但它修不了分布漂移后的失效，也修不了按类别的系统性偏差。
- **小样本 + 极端比例下，正态置信区间会越界，Wilson 区间更稳；** A/B 测试要按
  「先查 SRM → 再查功效 → 再查多重比较/偷看 → 最后才看 p 值」的顺序排查，不能跳步。
- **辛普森悖论/幸存者偏差/回归到均值/p-hacking 都有可复现的最小构造**——
  背下名字不够，要能在自己的数据里找出对应的机制并给出检测方法。

下一站：**模块 05 · 快问快答题库与自测** —— 整门课的收官，
150+ 题分主题题库、面试当天 90 分钟复习清单，以及一个真正能用的间隔重复自测引擎。